# Third level (group): ROI models, whole brain, figures, tables, brain maps

**ROI analyses** — linear mixed models on the run-level ROI values (standardized within ROI and event), random intercept per participant:

| RQ | Model (per event) |
|---|---|
| RQ1 group / imbalance | `beta_z ~ grp * system_c + drop + mag + covariates` (all 5 ROIs; + run variance component) |
| RQ2 reward content | per ROI: `beta_z ~ drop * mag + nev + covariates` |
| RQ3 person × content | per ROI: `beta_z ~ grp * drop * mag + nev + covariates` |

Coding: `drop`, `mag` = ±0.5 (variable = +0.5); `grp` 0 = typical, 1 = at risk; `system_c` reward = +0.5, control = −0.5;
`nev` = usable events in the run. FDR (Benjamini–Hochberg) across the five ROIs.

**Whole brain** — FSL `randomise` (TFCE, 5000 permutations) on the participant maps from notebook 3.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection
from scipy import stats, ndimage
from nibabel.affines import apply_affine
from nilearn import datasets, image, plotting

ROOT = Path("/srv/lab/fmri/crave")
ROI_DIR = ROOT / "derivatives" / "roi" / "VS_bold-tedana_fwhm-0_chest-block_rev-6"
WB_DIR = ROOT / "derivatives" / "wholebrain" / "VS_bold-tedana_fwhm-6_chest-block_rev-6"
OUT = ROOT / "derivatives" / "results"
OUT.mkdir(parents=True, exist_ok=True)

COVARIATES = "male + gaming_log + fd"     # "fd" = version without sex and gaming time
GROUP_COL = "group"                       # at risk at either survey; "group_pre" / "group_post" = sensitivity
EVENTS = ["anticipation", "reveal_all", "reveal_3", "boss_kill"]
MAIN_ROIS = ["vs_nacc", "ofc", "ant_ins", "dlpfc", "dacc"]
CONTROL_ROIS = ["wm", "csf", "v1", "motor", "global"]
NAMES = {"vs_nacc": "VS/NAcc", "ofc": "OFC", "ant_ins": "Anterior insula", "dlpfc": "dlPFC", "dacc": "dACC",
         "wm": "White matter", "csf": "Ventricles", "v1": "V1", "motor": "Precentral", "global": "Whole-brain mean"}
CELLS = {"FixedDrop-FixedMagnitude": "FF", "FixedDrop-VariableMagnitude": "FV",
         "VariableDrop-FixedMagnitude": "VF", "VariableDrop-VariableMagnitude": "VV"}

## 1. Prepare the ROI data

In [ ]:
roi = pd.read_csv(ROI_DIR / "roi_values_long.csv", dtype={"sub": str})

# in fixed-size rounds every reveal has 3 items, so reveal_all = reveal_3 there
key = ["sub", "run", "roi"]
r3 = roi[roi["contrast"] == "reveal_3"]
has_all = roi[roi["contrast"] == "reveal_all"].set_index(key).index
roi = pd.concat([roi, r3[~r3.set_index(key).index.isin(has_all)].assign(contrast="reveal_all")], ignore_index=True)

d = roi[roi["contrast"].isin(EVENTS) & roi["condition"].isin(CELLS)].copy()
d = d[d["beta"].notna() & (d["coverage"] >= 0.7) & (d["n_usable"] >= 1)]

d["grp"] = d[GROUP_COL].map({"typical": 0, "at_risk": 1})
d["drop"] = np.where(d["drop_schedule"] == "Variable", 0.5, -0.5)
d["mag"] = np.where(d["magnitude_schedule"] == "Variable", 0.5, -0.5)
d["system_c"] = np.where(d["system"] == "reward", 0.5, -0.5)
d["male"] = (d["sex"] == "male").astype(float)
d["gaming_log"] = np.log1p(d["gaming_h_per_day"])
d["gaming_log"] = d["gaming_log"].fillna(d["gaming_log"].median())
d["fd"] = d["mean_fd_filt"].fillna(d["mean_fd_filt"].median())
d["nev"] = d["n_usable"]
d["pos"] = d["run"].str[-2:].astype(int)                         # run position (sensitivity)
d["cell"] = d["condition"].map(CELLS)

gas = d.drop_duplicates("sub").set_index("sub")["gas7_mean"]
d["risk_z"] = d["sub"].map((gas - gas.mean()) / gas.std())        # continuous GAS-7

# standardize within ROI and event; the run's whole-brain mean is used in the robustness checks
d["beta_z"] = d.groupby(["contrast", "roi"])["beta"].transform(lambda x: (x - x.mean()) / x.std())
glob = d[d["roi"] == "global"].set_index(["sub", "run", "contrast"])["beta_z"]
d["glob_z"] = d.set_index(["sub", "run", "contrast"]).index.map(glob)

main = d[d["roi"].isin(MAIN_ROIS)]
print(d.groupby(GROUP_COL)["sub"].nunique())

In [ ]:
def fit(formula, data, run_variance=False):
    vc = {"run": "0 + C(run)"} if run_variance else None
    return smf.mixedlm(formula, data, groups=data["sub"], vc_formula=vc).fit(reml=True, method="lbfgs")

def terms(res, **info):
    names = res.fe_params.index
    ci = res.conf_int().loc[names]
    out = pd.DataFrame({"term": names, "estimate": res.fe_params.values, "se": res.bse_fe.values,
                        "z": res.tvalues[names].values, "p": res.pvalues[names].values,
                        "ci_low": ci[0].values, "ci_high": ci[1].values, "n_rows": res.nobs})
    for k, v in info.items():
        out[k] = v
    return out

def add_fdr(table):
    table["q_fdr"] = table.groupby(["model", "contrast", "term"])["p"].transform(lambda p: fdrcorrection(p)[1])
    return table

## 2. RQ1: group differences and reward/control imbalance

In [ ]:
rq1 = []
for event in EVENTS:
    e = main[(main["contrast"] == event) & main["grp"].notna()]
    rq1.append(terms(fit(f"beta_z ~ grp * system_c + drop + mag + {COVARIATES}", e, True), model="Group x System", contrast=event))
    rq1.append(terms(fit(f"beta_z ~ grp + C(roi) + drop + mag + {COVARIATES}", e[e["system"] == "reward"], True), model="Group, reward ROIs", contrast=event))
    rq1.append(terms(fit(f"beta_z ~ grp + C(roi) + drop + mag + {COVARIATES}", e[e["system"] == "control"], True), model="Group, control ROIs", contrast=event))
    rq1.append(terms(fit(f"beta_z ~ risk_z * system_c + drop + mag + {COVARIATES}", e, True), model="GAS-7 x System", contrast=event))
rq1 = pd.concat(rq1)
rq1 = rq1[rq1["term"].isin(["grp", "grp:system_c", "risk_z", "risk_z:system_c"])]
rq1.to_csv(OUT / "rq1_results.csv", index=False)
rq1[["contrast", "model", "term", "estimate", "se", "z", "p"]].round(3)

## 3. RQ2: reward content (drop rate × chest size)

In [ ]:
rq2 = []
for event in EVENTS:
    for r in MAIN_ROIS:
        e = main[(main["contrast"] == event) & (main["roi"] == r)]
        rq2.append(terms(fit(f"beta_z ~ drop * mag + nev + {COVARIATES}", e), model="RQ2", contrast=event, roi=r))
rq2 = pd.concat(rq2)
rq2 = add_fdr(rq2[rq2["term"].isin(["drop", "mag", "drop:mag"])])
rq2.to_csv(OUT / "rq2_results.csv", index=False)
rq2.pivot_table(index=["contrast", "roi"], columns="term", values="estimate").round(2)

## 4. RQ3: person × content

In [ ]:
rq3 = []
for event in EVENTS:
    for r in MAIN_ROIS:
        e = main[(main["contrast"] == event) & (main["roi"] == r)]
        rq3.append(terms(fit(f"beta_z ~ grp * drop * mag + nev + {COVARIATES}", e[e["grp"].notna()]), model="RQ3 group", contrast=event, roi=r))
        rq3.append(terms(fit(f"beta_z ~ risk_z * drop * mag + nev + {COVARIATES}", e), model="RQ3 GAS-7", contrast=event, roi=r))
rq3 = pd.concat(rq3)
rq3 = add_fdr(rq3[rq3["term"].isin(["grp", "grp:drop", "grp:mag", "grp:drop:mag", "risk_z:drop", "risk_z:mag", "risk_z:drop:mag"])])
rq3.to_csv(OUT / "rq3_results.csv", index=False)
rq3[rq3["term"].str.endswith("drop:mag")][["contrast", "roi", "model", "estimate", "se", "p", "q_fdr"]].round(3)

## 5. Sensitivity/Robustness checks

- **Negative-control regions** (white matter, ventricles, V1, precentral, whole-brain mean): an effect there means artefact or a brain-wide shift
- **Whole-brain-mean adjustment**: add the run's whole-brain mean as a covariate
- **Run position** as a covariate (only 9 of 24 counterbalancing orders were used)
- **Other group definitions**: pre-scan only, post-scan only

In [ ]:
checks = []
for event in ["anticipation", "reveal_all"]:
    for r in MAIN_ROIS + CONTROL_ROIS:
        e = d[(d["contrast"] == event) & (d["roi"] == r)]
        checks.append(terms(fit(f"beta_z ~ drop * mag + nev + {COVARIATES}", e), model="RQ2 unadjusted", contrast=event, roi=r))
        checks.append(terms(fit(f"beta_z ~ grp * drop * mag + nev + {COVARIATES}", e[e["grp"].notna()]), model="RQ3 unadjusted", contrast=event, roi=r))
        if r in MAIN_ROIS:
            checks.append(terms(fit(f"beta_z ~ drop * mag + nev + glob_z + {COVARIATES}", e), model="RQ2 adjusted", contrast=event, roi=r))
            checks.append(terms(fit(f"beta_z ~ grp * drop * mag + nev + glob_z + {COVARIATES}", e[e["grp"].notna()]), model="RQ3 adjusted", contrast=event, roi=r))
            checks.append(terms(fit(f"beta_z ~ grp * drop * mag + nev + glob_z + pos + {COVARIATES}", e[e["grp"].notna()]), model="RQ3 adjusted + run position", contrast=event, roi=r))
checks = pd.concat(checks)
checks = checks[checks["term"].isin(["mag", "grp:drop:mag"])]
checks.to_csv(OUT / "robustness_checks.csv", index=False)

checks[(checks["contrast"] == "anticipation") & (checks["term"] == "grp:drop:mag")].pivot_table(
    index="roi", columns="model", values="estimate").reindex(MAIN_ROIS + CONTROL_ROIS).round(2)

In [ ]:
# other group definitions (anticipation three-way)
other = []
for col in ["group_pre", "group_post"]:
    for r in MAIN_ROIS:
        e = main[(main["contrast"] == "anticipation") & (main["roi"] == r)].copy()
        e["grp"] = e[col].map({"typical": 0, "at_risk": 1})
        other.append(terms(fit(f"beta_z ~ grp * drop * mag + nev + {COVARIATES}", e[e["grp"].notna()]), model=col, contrast="anticipation", roi=r))
other = pd.concat(other)
other = add_fdr(other[other["term"] == "grp:drop:mag"])
other[["model", "roi", "estimate", "se", "p", "q_fdr"]].round(3)

## 6. Figures

In [ ]:
# heat map of z values: group effect (RQ1), size and drop (RQ2), group x drop x size (RQ3)
panels = [("RQ1: group", rq3, "RQ3 group", "grp"),
          ("RQ2: chest size", rq2, "RQ2", "mag"),
          ("RQ2: drop rate", rq2, "RQ2", "drop"),
          ("RQ3: group x drop x size", rq3, "RQ3 group", "grp:drop:mag")]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (title, table, model, term) in zip(axes, panels):
    t = table[(table["model"] == model) & (table["term"] == term)]
    z = t.pivot(index="roi", columns="contrast", values="z").reindex(index=MAIN_ROIS, columns=EVENTS)
    q = t.pivot(index="roi", columns="contrast", values="q_fdr").reindex(index=MAIN_ROIS, columns=EVENTS)
    ax.imshow(z, cmap="RdBu_r", vmin=-4, vmax=4)
    for i in range(len(MAIN_ROIS)):
        for j in range(len(EVENTS)):
            star = "**" if q.iloc[i, j] < 0.05 else ""
            ax.text(j, i, f"{z.iloc[i, j]:.1f}{star}", ha="center", va="center", fontsize=9)
    ax.set_xticks(range(len(EVENTS)))
    ax.set_xticklabels(EVENTS, rotation=40, ha="right")
    ax.set_yticks(range(len(MAIN_ROIS)))
    ax.set_yticklabels([NAMES[r] for r in MAIN_ROIS])
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUT / "fig_roi_heatmap.png", dpi=300)
plt.show()

In [ ]:
# RQ2 chest-size effect for item reveals: a priori vs control regions, unadjusted vs adjusted
f = checks[(checks["contrast"] == "reveal_all") & (checks["term"] == "mag")]
order = MAIN_ROIS + CONTROL_ROIS
fig, ax = plt.subplots(figsize=(7, 5))
for model, color, shift in [("RQ2 unadjusted", "#1f4e79", -0.15), ("RQ2 adjusted", "#c55a11", 0.15)]:
    s = f[f["model"] == model].set_index("roi").reindex(order)
    y = np.arange(len(order)) + shift
    ax.errorbar(s["estimate"], y, xerr=[s["estimate"] - s["ci_low"], s["ci_high"] - s["estimate"]],
                fmt="o", color=color, capsize=2, label=model.replace("RQ2 ", "").replace("adjusted", "adjusted for whole-brain mean"))
ax.axvline(0, color="black", lw=0.8)
ax.axhline(4.5, color="grey", ls="--", lw=0.8)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([NAMES[r] for r in order])
ax.invert_yaxis()
ax.set_xlabel("Variable - fixed chest size (SD units, 95% CI)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "fig_rq2_forest.png", dpi=300)
plt.show()

In [ ]:
# RQ3: anticipation by condition and group (participant means, z)
a = d[(d["contrast"] == "anticipation") & d["grp"].notna()]
cell_means = a.groupby(["roi", GROUP_COL, "cell", "sub"])["beta_z"].mean().groupby(["roi", GROUP_COL, "cell"]).agg(["mean", "sem"]).reset_index()
cell_means.to_csv(OUT / "rq3_cell_means.csv", index=False)

rois = MAIN_ROIS + ["global"]
fig, axes = plt.subplots(1, len(rois), figsize=(18, 3.5), sharey=True)
for ax, r in zip(axes, rois):
    for grp, color in [("typical", "#1f4e79"), ("at_risk", "#c55a11")]:
        s = cell_means[(cell_means["roi"] == r) & (cell_means[GROUP_COL] == grp)].set_index("cell").reindex(["FF", "FV", "VF", "VV"])
        ax.errorbar(range(4), s["mean"], yerr=s["sem"], fmt="o-", color=color, capsize=2, label=grp)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_xticks(range(4))
    ax.set_xticklabels(["FF", "FV", "VF", "VV"])
    ax.set_title(NAMES[r])
axes[0].set_ylabel("Anticipation (z)")
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "fig_rq3_cells.png", dpi=300)
plt.show()

## 7. Whole brain: randomize

Runs every analysis written by notebook 3 (slow: start it in `screen`/`tmux` from a terminal if you prefer:
`bash run_randomise.sh`). Outputs: `rand_<design>_tstat1/2` and `rand_<design>_tfce_corrp_tstat1/2` (1 − FWE p).

In [ ]:
!bash {WB_DIR}/run_randomise.sh

In [ ]:
# cluster table: TFCE-corrected p < .05, clusters of >= 10 voxels, peak MNI coordinate and atlas label
ho = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
ref = nib.load(str(WB_DIR / "mask.nii.gz"))
ho_data = np.asarray(image.resample_to_img(ho.maps, ref, interpolation="nearest").dataobj).astype(int)

rows = []
for tmap in sorted(WB_DIR.glob("*/rand_*_tstat[12].nii.gz")):
    if "tfce" in tmap.name:
        continue
    corrp = Path(str(tmap).replace("_tstat", "_tfce_corrp_tstat"))
    t = nib.load(str(tmap)).get_fdata()
    p_fwe = 1 - nib.load(str(corrp)).get_fdata()
    labels, n = ndimage.label(p_fwe < 0.05, structure=np.ones((3, 3, 3)))
    rows.append({"analysis": tmap.parent.name, "map": tmap.name, "cluster": 0, "voxels": 0, "min_p_fwe": p_fwe.min()})
    for c in range(1, n + 1):
        in_cluster = labels == c
        if in_cluster.sum() < 10:
            continue
        peak = np.unravel_index(np.argmax(np.where(in_cluster, np.abs(t), 0)), t.shape)
        x, y, z = apply_affine(ref.affine, peak)
        rows.append({"analysis": tmap.parent.name, "map": tmap.name, "cluster": c, "voxels": int(in_cluster.sum()),
                     "peak_t": t[peak], "x": x, "y": y, "z": z, "min_p_fwe": p_fwe[in_cluster].min(),
                     "label": ho.labels[ho_data[peak]]})
clusters = pd.DataFrame(rows)
clusters.to_csv(OUT / "wholebrain_clusters.csv", index=False)
clusters[clusters["cluster"] > 0] if (clusters["cluster"] > 0).any() else clusters.sort_values("min_p_fwe").head(10)

In [ ]:
# brain maps for each RQ: corrected clusters if any, otherwise the uncorrected t-map at p < .001 (labelled)
template = datasets.load_mni152_template(resolution=1)

def brain_map(analysis, design, tstat, title, ax):
    folder = WB_DIR / analysis
    t = nib.load(str(folder / f"rand_{design}_tstat{tstat}.nii.gz"))
    p_fwe = 1 - nib.load(str(folder / f"rand_{design}_tfce_corrp_tstat{tstat}.nii.gz")).get_fdata()
    if (p_fwe < 0.05).any():
        shown = np.where(p_fwe < 0.05, t.get_fdata(), 0)
        title = title + "  (TFCE p < .05)"
    else:
        df = len(pd.read_csv(folder / "subjects.csv")) - 4
        shown = np.where(t.get_fdata() > stats.t.ppf(0.999, df), t.get_fdata(), 0)
        title = title + "  (n.s. after correction; shown: p < .001 uncorrected)"
    plotting.plot_stat_map(nib.Nifti1Image(shown, t.affine), bg_img=template, display_mode="z",
                           cut_coords=[-8, 0, 8, 16, 32, 44], threshold=0.01, vmax=6, title=title, axes=ax)

brain_maps = {
    "RQ1": [("anticipation_main", "group", 1, "Anticipation: at risk > typical"),
            ("anticipation_main", "group", 2, "Anticipation: typical > at risk"),
            ("reveal_all_main", "group", 1, "Reveal: at risk > typical")],
    "RQ2": [("reveal_all_mag", "onesample", 1, "Reveal: variable > fixed size"),
            ("reveal_all_drop", "onesample", 1, "Reveal: 60% > 100% drop"),
            ("anticipation_mag", "onesample", 1, "Anticipation: variable > fixed size")],
    "RQ3": [("anticipation_pmx", "group", 1, "Anticipation pure - mixed: at risk > typical"),
            ("anticipation_pmx", "group", 2, "Anticipation pure - mixed: typical > at risk")],
    "Task": [("anticipation_main", "onesample", 1, "Chest anticipation"),
             ("reveal_all_main", "onesample", 1, "Item reveal"),
             ("boss_kill_main", "onesample", 1, "Boss kill")],
}
for rq, rows_ in brain_maps.items():
    fig, axes = plt.subplots(len(rows_), 1, figsize=(14, 2.6 * len(rows_)))
    for ax, (analysis, design, tstat, title) in zip(np.atleast_1d(axes), rows_):
        brain_map(analysis, design, tstat, title, ax)
    fig.savefig(OUT / f"fig_brain_{rq}.png", dpi=200)
    plt.show()

## 8. Tables

In [ ]:
# participant characteristics by group (Welch t tests)
people = d.drop_duplicates("sub")
rows = []
for col in ["age", "gas7_pre", "gas7_post", "vgaq_pre", "vgaq_post", "iat_pre", "iat_post", "gaming_h_per_day"]:
    a = people.loc[people[GROUP_COL] == "at_risk", col].dropna()
    b = people.loc[people[GROUP_COL] == "typical", col].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({"measure": col, "at risk M (SD)": f"{a.mean():.2f} ({a.std():.2f})",
                 "typical M (SD)": f"{b.mean():.2f} ({b.std():.2f})", "t": round(t, 2), "p": round(p, 3)})
table_sample = pd.DataFrame(rows)
table_sample.to_csv(OUT / "table_sample.csv", index=False)
print(people[GROUP_COL].value_counts().to_dict(), "| female:", people.groupby(GROUP_COL)["sex"].apply(lambda s: (s == "female").sum()).to_dict())
table_sample

In [ ]:
# data quality by group (participant means of the runs in the models)
runs = d.drop_duplicates(["sub", "run"])
quality = runs.groupby(["sub", GROUP_COL]).agg(runs=("run", "size"), mean_fd_filt=("mean_fd_filt", "mean")).reset_index()
quality.groupby(GROUP_COL)[["runs", "mean_fd_filt"]].agg(["mean", "std"]).round(3)

In [ ]:
# one results table per RQ (beta (SE), q) for the write-up
def fmt(t):
    return t["estimate"].map("{:.2f}".format) + " (" + t["se"].map("{:.2f}".format) + ")"

table_rq1 = rq1.assign(value=fmt(rq1), p=rq1["p"].round(3))[["contrast", "model", "term", "value", "p"]]
table_rq2 = rq2.assign(value=fmt(rq2)).pivot_table(index=["contrast", "roi"], columns="term", values="value", aggfunc="first")
table_rq3 = rq3[rq3["term"].str.endswith("drop:mag")].assign(value=fmt).pivot_table(
    index=["contrast", "roi"], columns="model", values="value", aggfunc="first")
table_rq1.to_csv(OUT / "table_rq1.csv", index=False)
table_rq2.to_csv(OUT / "table_rq2.csv")
table_rq3.to_csv(OUT / "table_rq3.csv")
table_rq3